<a href="https://colab.research.google.com/github/Rishii077/AI-Lab-Assignments/blob/main/Experiment_2_RAG_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 14.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.


In [3]:
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

print("Gemini connected successfully!")

Gemini connected successfully!


In [4]:
document = """
Artificial Intelligence (AI) is a branch of computer science that focuses
on creating systems capable of performing tasks that normally require
human intelligence.

Machine Learning (ML) is a subset of Artificial Intelligence. It allows
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every task.

Deep Learning is a subset of Machine Learning that uses artificial neural
networks with multiple layers. Deep learning is commonly used in image
recognition, speech recognition and natural language processing.

Natural Language Processing (NLP) is a field of Artificial Intelligence
that enables computers to understand, process and generate human language.

Generative AI refers to AI systems that can create new content such as
text, images, audio and code.

Retrieval-Augmented Generation (RAG) combines information retrieval with
generative AI. A RAG system first retrieves relevant information from an
external knowledge source and then provides that information to a
language model to generate a grounded answer.

Embeddings are numerical representations of text. They allow similar
pieces of text to be compared using mathematical similarity measures.

RAG systems commonly contain three major stages: indexing, retrieval,
and response generation. During indexing, documents are divided into
smaller chunks and converted into embeddings. During retrieval, the
system finds the chunks most relevant to a user's question. During
response generation, a language model uses the retrieved information
to produce the final answer.
"""

print("Knowledge document created successfully!")

Knowledge document created successfully!


In [5]:
def create_chunks(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)

    return chunks


chunks = create_chunks(document)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk)

Number of chunks: 4

--- Chunk 1 ---

Artificial Intelligence (AI) is a branch of computer science that focuses
on creating systems capable of performing tasks that normally require
human intelligence.

Machine Learning (ML) is a subset of Artificial Intelligence. It allows
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every task.

Deep Learning is a subset of Machine Learning that uses artificial neural
networks with multiple layers. Deep learning is commonly used i

--- Chunk 2 ---
n image
recognition, speech recognition and natural language processing.

Natural Language Processing (NLP) is a field of Artificial Intelligence
that enables computers to understand, process and generate human language.

Generative AI refers to AI systems that can create new content such as
text, images, audio and code.

Retrieval-Augmented Generation (RAG) combines information retrieval with
generative AI. A RAG system first retrieves rele

In [6]:
embeddings = []

for chunk in chunks:

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=chunk
    )

    embeddings.append(result.embeddings[0].values)

print("Embeddings created successfully!")
print("Number of embeddings:", len(embeddings))

Embeddings created successfully!
Number of embeddings: 4


In [7]:
import numpy as np

print("NumPy imported successfully!")

NumPy imported successfully!


In [8]:
def cosine_similarity(a, b):

    a = np.array(a)
    b = np.array(b)

    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

print("Cosine similarity function created successfully!")

Cosine similarity function created successfully!


In [9]:
def retrieve_relevant_chunks(question, top_k=2):

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=question
    )

    question_embedding = result.embeddings[0].values

    similarities = []

    for i, embedding in enumerate(embeddings):

        score = cosine_similarity(
            question_embedding,
            embedding
        )

        similarities.append((i, score))

    similarities.sort(
        key=lambda x: x[1],
        reverse=True
    )

    selected_chunks = []

    for index, score in similarities[:top_k]:
        selected_chunks.append(
            (chunks[index], score)
        )

    return selected_chunks

In [10]:
question = "What is Retrieval-Augmented Generation?"

retrieved = retrieve_relevant_chunks(question)

print("Question:")
print(question)

print("\nRetrieved Information:")

for i, (chunk, score) in enumerate(retrieved):

    print(f"\n--- Retrieved Chunk {i + 1} ---")
    print("Similarity Score:", score)
    print(chunk)

Question:
What is Retrieval-Augmented Generation?

Retrieved Information:

--- Retrieved Chunk 1 ---
Similarity Score: 0.778607036124814
then provides that information to a
language model to generate a grounded answer.

Embeddings are numerical representations of text. They allow similar
pieces of text to be compared using mathematical similarity measures.

RAG systems commonly contain three major stages: indexing, retrieval,
and response generation. During indexing, documents are divided into
smaller chunks and converted into embeddings. During retrieval, the
system finds the chunks most relevant to a user's question. During
res

--- Retrieved Chunk 2 ---
Similarity Score: 0.7697141261772629
n image
recognition, speech recognition and natural language processing.

Natural Language Processing (NLP) is a field of Artificial Intelligence
that enables computers to understand, process and generate human language.

Generative AI refers to AI systems that can create new content such as
text,

In [11]:
def rag_question_answer(question):

    retrieved = retrieve_relevant_chunks(question)

    context = "\n\n".join(
        chunk for chunk, score in retrieved
    )

    prompt = f"""
You are a helpful AI assistant.

Answer the user's question using ONLY the information
provided in the context below.

Context:
{context}

User Question:
{question}

Give a clear and simple answer.

If the answer is not available in the context, say:
"Information not available in the provided document."
"""

    interaction = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    answer = interaction.output_text.strip()

    print("========== RAG QUESTION ANSWERING ==========")

    print("\nQuestion:")
    print(question)

    print("\nRetrieved Context:")
    print(context)

    print("\nGenerated Answer:")
    print(answer)

In [12]:
rag_question_answer(
    "What is Retrieval-Augmented Generation?"
)

========== RAG QUESTION ANSWERING ==========

Question:
What is Retrieval-Augmented Generation?

Retrieved Context:
then provides that information to a
language model to generate a grounded answer.

Embeddings are numerical representations of text. They allow similar
pieces of text to be compared using mathematical similarity measures.

RAG systems commonly contain three major stages: indexing, retrieval,
and response generation. During indexing, documents are divided into
smaller chunks and converted into embeddings. During retrieval, the
system finds the chunks most relevant to a user's question. During
res

n image
recognition, speech recognition and natural language processing.

Natural Language Processing (NLP) is a field of Artificial Intelligence
that enables computers to understand, process and generate human language.

Generative AI refers to AI systems that can create new content such as
text, images, audio and code.

Retrieval-Augmented Generation (RAG) combines information 

In [13]:
rag_question_answer(
    "What is quantum computing?"
)

========== RAG QUESTION ANSWERING ==========

Question:
What is quantum computing?

Retrieved Context:

Artificial Intelligence (AI) is a branch of computer science that focuses
on creating systems capable of performing tasks that normally require
human intelligence.

Machine Learning (ML) is a subset of Artificial Intelligence. It allows
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every task.

Deep Learning is a subset of Machine Learning that uses artificial neural
networks with multiple layers. Deep learning is commonly used i

n image
recognition, speech recognition and natural language processing.

Natural Language Processing (NLP) is a field of Artificial Intelligence
that enables computers to understand, process and generate human language.

Generative AI refers to AI systems that can create new content such as
text, images, audio and code.

Retrieval-Augmented Generation (RAG) combines information retrieval wit